<a href="https://colab.research.google.com/github/Mohammed-Ahmed-Alansi/Applied-ML-Projects/blob/main/Day2_HOML_Housing_Instructor_AR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2 — Applied Machine Learning
## Data Preparation, Leakage, Pipelines, and Model Benchmarking

**Primary dataset:** the `housing.csv` dataset used with *Hands-On Machine Learning*.

**Book alignment:** Chapter 2 — End-to-End Machine Learning Project.

**Core workflow**
`Raw Data → Stratified Split → Preprocessing → Pipeline → Models → Evaluation`

**Modern extension**
We also benchmark `HistGradientBoostingRegressor`, `XGBRegressor`, and `LGBMRegressor` using the same preprocessing contract.

### Important
This notebook is the instructor's executable application notebook.  
All code explanations are written in **English** so the notebook can also be used as a technical reference.

## Learning Targets for the Session

By the end of the notebook, learners should be able to:

1. Split the housing data using the book-style income stratification.
2. Detect missing values and mixed data types.
3. Build numeric and categorical preprocessing pipelines.
4. Use `SimpleImputer`, `StandardScaler`, and `OneHotEncoder`.
5. Combine transformations with `ColumnTransformer`.
6. Build one leakage-safe `Pipeline` from raw features to predictions.
7. Benchmark multiple regression algorithms on the same prepared data.
8. Explain why XGBoost and LightGBM are useful modern tree-boosting extensions.
9. Demonstrate a real target/data leakage problem.
10. Produce an evidence-based model comparison.

# 0. Environment Setup

### Why are we doing this?

We want the same notebook to run in Google Colab and to demonstrate both the classical scikit-learn models from the book and modern gradient-boosting libraries.

For the core lesson, we use the scikit-learn estimator interface so different models can share the same preprocessing pipeline.

In [ ]:
%pip -q install -U xgboost lightgbm

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

print("Environment ready.")

# 1. Load the Hands-On ML Housing Dataset

The book's dataset repository provides `housing.csv` inside the official `ageron/data` repository.

**Official raw file:**
`https://raw.githubusercontent.com/ageron/data/main/housing/housing.csv`

The dataset contains the missing `total_bedrooms` values used in the book to introduce data cleaning and preprocessing.

In [ ]:
HOUSING_URL = "https://raw.githubusercontent.com/ageron/data/main/housing/housing.csv"

housing = pd.read_csv(HOUSING_URL)

print("Shape:", housing.shape)
housing.head()

In [ ]:
print(housing.info())

print("\nMissing values:")
display(housing.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

print("\nCategorical columns:")
print(housing.select_dtypes(include="object").columns.tolist())

# 2. Quick Problem Framing

### Business question

> Can we predict the median house value for a district from the district's available characteristics?

### Target
`median_house_value`

### Features
All columns except the target.

### Problem type
Supervised Learning → Regression.

We will keep this framing visible because preprocessing choices only make sense in relation to the prediction task.

# 3. Hands-On ML Style Stratified Split

The book introduces an `income_cat` variable from `median_income` and uses stratified sampling so the train and test sets preserve the income distribution more reliably than a purely random split.

### Teaching point

The split comes **before fitting any data-dependent preprocessing step**.

The stratification variable is a feature-derived sampling aid, not a target-derived feature.

In [ ]:
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5]
)

print(housing["income_cat"].value_counts(normalize=True).sort_index())

In [ ]:
splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

for train_idx, test_idx in splitter.split(housing, housing["income_cat"]):
    strat_train_set = housing.iloc[train_idx].copy()
    strat_test_set = housing.iloc[test_idx].copy()

strat_train_set.drop(columns=["income_cat"], inplace=True)
strat_test_set.drop(columns=["income_cat"], inplace=True)

print("Train shape:", strat_train_set.shape)
print("Test shape :", strat_test_set.shape)

# 4. Separate Features and Target

We remove the target from `X` and keep it in `y`.

We also remove `income_cat` because it was created only to control the split and should not be used as a model feature.

In [ ]:
TARGET = "median_house_value"

X_train = strat_train_set.drop(columns=[TARGET])
y_train = strat_train_set[TARGET].copy()

X_test = strat_test_set.drop(columns=[TARGET])
y_test = strat_test_set[TARGET].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

# 5. Identify Numeric and Categorical Features

The housing data contains:
- numeric measurements such as `total_rooms` and `median_income`
- one categorical feature: `ocean_proximity`
- missing values in `total_bedrooms`

Different feature types require different preprocessing steps.

In [ ]:
numeric_features = X_train.select_dtypes(
    include=[np.number]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# 6. Numeric Preprocessing Pipeline

We will:
1. Impute missing numeric values using the training median.
2. Standardize numeric features.

### Why median?

The median is less sensitive than the mean to extreme values.

### Why scaling?

Scaling is important for many models, especially linear models and distance-based methods. Tree-based models generally do not require scaling, but keeping preprocessing encapsulated makes the workflow consistent and teaches the reusable pipeline pattern.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

numeric_pipeline

# 7. Categorical Preprocessing Pipeline

We will:
1. Impute missing categorical values using the most frequent category.
2. One-hot encode categories.
3. Ignore unseen categories at prediction time.

`handle_unknown="ignore"` is important because production data can contain a category that did not appear in the training set.

In [ ]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )),
])

categorical_pipeline

# 8. Combine the Two Pipelines with ColumnTransformer

`ColumnTransformer` lets us apply different transformations to different columns while keeping the whole operation as one estimator.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

preprocessor

# 9. Build the End-to-End Pipeline

This is the critical engineering step.

Instead of manually:
- fitting an imputer,
- transforming the train set,
- transforming the test set,
- encoding categories,
- scaling numbers,
- and then fitting a model,

we attach preprocessing and the model to one `Pipeline`.

When `fit` is called on the pipeline, preprocessing is learned from the training data and the transformed data is passed to the model.

In [ ]:
linear_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

linear_pipeline.fit(X_train, y_train)

linear_pred = linear_pipeline.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_r2 = r2_score(y_test, linear_pred)

print(f"Linear Regression MAE : {linear_mae:,.3f}")
print(f"Linear Regression RMSE: {linear_rmse:,.3f}")
print(f"Linear Regression R²  : {linear_r2:.3f}")

# 10. Verify the Transformed Feature Space

This is a good place to show learners that categorical encoding increases the feature count.

The model does not receive the raw `ocean_proximity` strings.  
It receives a numeric matrix produced by the preprocessing pipeline.

In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train)

print("Raw feature count       :", X_train.shape[1])
print("Transformed feature count:", X_train_transformed.shape[1])

# 11. Benchmark Multiple Algorithms

The book introduces classical models such as:
- Linear Regression
- Decision Tree
- Random Forest

We will extend the same pipeline pattern to:
- Ridge Regression
- Extra Trees
- HistGradientBoosting
- XGBoost
- LightGBM

### Teaching rule

We are not trying to "teach nine algorithms" in one session.

We are teaching a more powerful pattern:

> **One data contract → many candidate estimators → comparable experiments.**

XGBoost exposes a scikit-learn-compatible estimator interface, and LightGBM provides `LGBMRegressor`, so both can participate in the same experiment structure.

In [ ]:
models = {
    "Dummy Baseline": DummyRegressor(strategy="median"),
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=10.0),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=None,
        random_state=42
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        random_state=42
    ),
    "XGBoost": XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    ),
}

In [ ]:
results = []

for name, estimator in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])

    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Pipeline": pipe,
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

results_df[["Model", "MAE", "RMSE", "R2"]]

# 12. Read the Benchmark Correctly

Do not announce "the winner" from one number.

Ask:
- Did the model beat the baseline?
- Is the improvement meaningful?
- What is the trade-off in complexity and training time?
- Is the model reproducible?
- Is the preprocessing identical?
- Are we still evaluating only once on the test set?

The test set should remain a final check rather than a tuning playground.

In [ ]:
results_df[["Model", "MAE", "RMSE", "R2"]].style.format({
    "MAE": "{:,.3f}",
    "RMSE": "{:,.3f}",
    "R2": "{:.3f}",
})

# 13. Modern Model Comparison — Why XGBoost and LightGBM?

### XGBoost
A gradient-boosted tree system with a scikit-learn-compatible estimator interface.

### LightGBM
A gradient boosting framework optimized for efficient training and a scikit-learn-compatible `LGBMRegressor`.

### Key teaching idea

Both are **candidate estimators**, not replacements for problem framing, preprocessing, validation, and error analysis.

A stronger algorithm does not rescue a leaked or badly framed experiment.

In [ ]:
modern_subset = results_df[
    results_df["Model"].isin([
        "Random Forest",
        "HistGradientBoosting",
        "XGBoost",
        "LightGBM"
    ])
][["Model", "MAE", "RMSE", "R2"]]

modern_subset

# 14. Unseen Category Demonstration

We will create one test row whose `ocean_proximity` value is a category that was not present during training.

Because the encoder uses `handle_unknown="ignore"`, prediction should still work.

This models an important production reality: the world can contain values the training set never saw.

In [ ]:
demo_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=10.0)),
])

demo_pipe.fit(X_train, y_train)

X_test_new_category = X_test.copy()
X_test_new_category.loc[X_test_new_category.index[0], "ocean_proximity"] = "NEW_CATEGORY"

new_category_prediction = demo_pipe.predict(
    X_test_new_category.iloc[[0]]
)

print("Prediction succeeded:", new_category_prediction[0])

# 15. Data Leakage Experiment — Wrong vs Correct

Now we deliberately create an invalid experiment.

### Wrong idea
Fit an imputer/scaler on the complete dataset before the train/test split.

That allows information from the future test set to influence the preprocessing statistics.

### Correct idea
Split first, then fit preprocessing only on training data.

A `Pipeline` helps enforce this pattern when it is used inside model selection and cross-validation.

In [ ]:
# WRONG ON PURPOSE — for demonstration only.
X_all = housing.drop(columns=[TARGET])
y_all = housing[TARGET]

wrong_numeric_imputer = SimpleImputer(strategy="median")

# This uses statistics from the complete dataset, including future test rows.
X_all_imputed = wrong_numeric_imputer.fit_transform(
    X_all[numeric_features]
)

print("This cell demonstrates a leakage-prone pattern.")

In [ ]:
# CORRECT PATTERN
safe_numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

safe_numeric_pipeline.fit(X_train[numeric_features])

train_ready = safe_numeric_pipeline.transform(X_train[numeric_features])
test_ready = safe_numeric_pipeline.transform(X_test[numeric_features])

print("Train transformed:", train_ready.shape)
print("Test transformed :", test_ready.shape)

# 16. Book-Style Feature Engineering Extension

Hands-On ML introduces useful derived ratios for the housing problem, such as:
- rooms per household
- population per household
- bedrooms per room

This is a good transition from "cleaning data" to "creating information".

For today's core lesson, the benchmark above keeps the pipeline simple.  
The next cell shows how the ratios can be added in a controlled transformation step.

In [ ]:
from sklearn.preprocessing import FunctionTransformer

def add_housing_ratios(X):
    X = X.copy()

    X["rooms_per_household"] = (
        X["total_rooms"] / X["households"]
    )
    X["population_per_household"] = (
        X["population"] / X["households"]
    )
    X["bedrooms_per_room"] = (
        X["total_bedrooms"] / X["total_rooms"]
    )

    return X

ratio_adder = FunctionTransformer(
    add_housing_ratios,
    feature_names_out="one-to-one"
)

ratio_demo = ratio_adder.fit_transform(
    X_train[[
        "total_rooms",
        "total_bedrooms",
        "population",
        "households"
    ]].copy()
)

ratio_demo.head()

# 17. Mini Challenge — Choose a Pipeline

### Scenario

Your team must ship a house-price model.

Constraints:
- The input is raw CSV data.
- Some numeric values are missing.
- `ocean_proximity` is categorical.
- New categories may appear in production.
- The model must be reproducible.

### Task

Choose one of the benchmark models and justify:
1. Why this estimator?
2. Why this preprocessing?
3. What leakage risk did your design remove?
4. What would you validate next?

# 18. Day 2 Artifact

Save the following:

1. `day2_housing_pipeline.ipynb`
2. `day2_model_comparison.csv`
3. `day2_leakage_notes.md`

### Definition of Done

- Train/test split is correct.
- Missing values are handled.
- Categorical data is encoded.
- Preprocessing is inside a Pipeline.
- At least 5 models are compared.
- XGBoost and LightGBM are included.
- The test set is not used to fit preprocessing.
- The learner can explain the final model choice.

In [ ]:
# Save the benchmark table as a reusable artifact.
results_df[["Model", "MAE", "RMSE", "R2"]].to_csv(
    "/content/day2_model_comparison.csv",
    index=False
)

print("Saved /content/day2_model_comparison.csv")